# DNA Modeling with Enformer (DeepMind)

<div style="text-align: center;">
    <img src="assets/enformer_model.png" width="60%">
</div>


Enformer is a state-of-the-art deep learning model developed by DeepMind for predicting gene regulation from DNA sequences. Leveraging transformer architectures, Enformer can accurately forecast chromatin accessibility, histone modifications, and transcription factor binding across the genome.

Here, we demonstrate how to use Enformer with GPU acceleration to efficiently predict regulatory features from DNA sequences and visualize the results in an embedded genome browser.

# Setup

In [ ]:
# requirements
!pip install -q kipoiseq pyfaidx igv-notebook tensorflow-hub

import tensorflow as tf
assert tf.config.list_physical_devices('GPU'), 'GPU required: Runtime -> Change runtime type -> GPU'

# Run Enformer Model

In [ ]:
# Run model
import tensorflow_hub as hub
import numpy as np
from utils.enformer import (
    download_genome, FastaExtractor, one_hot_encode,
    get_input_interval, get_output_region, export_tracks
)

# Configuration
CHROM = 'chr17'
CENTER = 7_580_000  # TP53 gene

# Prepare sequence
fasta_path = download_genome()
fasta = FastaExtractor(fasta_path)
interval = get_input_interval(CHROM, CENTER)
sequence = one_hot_encode(fasta.extract(interval))

print(f"Target: {CHROM}:{CENTER:,} | Input: {len(sequence):,} bp")

# Load model and run inference
model = hub.load('https://tfhub.dev/deepmind/enformer/1').model
predictions = model.predict_on_batch(sequence[np.newaxis])['human'].numpy()[0]

# Export tracks
!mkdir -p runs
track_files = export_tracks(predictions, CHROM, interval.start, './runs/tracks')

# Visualise Results

In [ ]:
# Visualize
from utils.igv_utils import create_browser, show_interpretation_guide

out_start, out_end = get_output_region(interval.start)
locus = f"{CHROM}:{out_start}-{out_end}"

browser = create_browser(locus, track_files)
show_interpretation_guide()